In [ ]:
import os

# Define your class-to-ID mapping based on filename prefixes
class_prefix_to_id = {
    "no_fall": 0,
    "fall": 1,
}

# Path to the directory where all .txt files are located (e.g., labels/train/)
labels_dir = "new_dataset/labels/val"  # Update this

# Loop through all files in the label directory
for filename in os.listdir(labels_dir):
    if filename.endswith(".txt"):
        for prefix, class_id in class_prefix_to_id.items():
            if filename.startswith(prefix):
                label_path = os.path.join(labels_dir, filename)

                with open(label_path, 'r') as f:
                    lines = f.readlines()

                # Replace class ID on each line
                updated_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        parts[0] = str(class_id)  # Replace the class ID
                        updated_lines.append(" ".join(parts) + "\n")

                # Save the updated label file
                with open(label_path, 'w') as f:
                    f.writelines(updated_lines)

                print(f"Updated {filename} → class ID: {class_id}")
                break  # Stop checking other prefixes once matched


In [ ]:
import os
from PIL import Image

# Set the folder where your images are stored
folder_path = "new_dataset/val/fall/images"  # ← change this to your actual folder

# Loop through all files in the folder
for filename in os.listdir(folder_path):
    if filename.lower().endswith(('.jpg', '.jpeg')):
        img_path = os.path.join(folder_path, filename)
        img = Image.open(img_path)

        # Remove extension and add .png
        base_name = os.path.splitext(filename)[0]
        new_filename = f"{base_name}.png"
        new_path = os.path.join(folder_path, new_filename)

        # Save as PNG
        img.save(new_path, "PNG")
        print(f"Converted: {filename} → {new_filename}")


In [ ]:
import os
import cv2

# Inisialisasi dataset
dataset_root = "new_dataset"
splits       = ["train", "val"]
class_names  = ["no_fall", "fall"]

# Pengulangan untuk semua data dalam folder
for split in splits:
    img_dir = os.path.join(dataset_root, "images", split)
    lbl_dir = os.path.join(dataset_root, "labels", split)
    out_dir = os.path.join(dataset_root, "visuals", split)
    os.makedirs(out_dir, exist_ok=True)

    if not os.path.isdir(img_dir):
        print(f"⚠️  Skipping missing folder: {img_dir}")
        continue

    # Mencari semua gambar
    for fn in os.listdir(img_dir):
        if not fn.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        img_path = os.path.join(img_dir, fn)
        lbl_path = os.path.join(lbl_dir, os.path.splitext(fn)[0] + ".txt")
        out_path = os.path.join(out_dir, fn)

        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️  Couldn't read image: {img_path}")
            continue

        h, w = img.shape[:2]

        # Menggambar bounding box
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cid, xc, yc, bw, bh = parts
                    cid = int(cid)
                    xc, yc, bw, bh = map(float, (xc, yc, bw, bh))

                    # Denormalisasi bentuk bounding box
                    x1 = int((xc - bw/2) * w)
                    y1 = int((yc - bh/2) * h)
                    x2 = int((xc + bw/2) * w)
                    y2 = int((yc + bh/2) * h)

                    color = (0,255,0) if cid==0 else (0,0,255)
                    label = class_names[cid]

                    cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
                    cv2.putText(img, label, (x1, y1-10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # Simpan hasil
        cv2.imwrite(out_path, img)
        print(f"✅  Saved {out_path}")
